# Прототип: Анализ видео маятника-шарика

Цель: вычислить ускорение свободного падения $g$ на основе анализа видео маятника

In [ ]:
import cv2
import pandas as pd
from IPython.display import Video

# Путь к видео файлу
video_path = "ball_pendulum.mp4"

# Проверим, что файл существует
import os
if os.path.exists(video_path):
    print(f"Видео найдено: {video_path}")
    # Получим информацию о видео
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    
    print(f"FPS: {fps}")
    print(f"Кадров: {frame_count}")
    print(f"Длительность: {duration:.2f} сек")
    print(f"Разрешение: {width}x{height}")
    print()
    
    # Воспроизведение видео
    display(Video(video_path, width=600))
else:
    print(f"Видео не найдено: {video_path}")
    print("Поместите видео в папку с notebook")

In [ ]:
import subprocess

# Параметры обрезки
start_sec = ...  # Начало в секундах
end_sec = -1     # Конец в секундах (-1 = до конца)

# Определяем длительность
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
cap.release()

if end_sec == -1:
    end_sec = duration

# Обрезка с помощью FFmpeg
trimmed_path = "ball_trimmed.mp4"
duration_cut = end_sec - start_sec

cmd = [
    'ffmpeg', '-y',
    '-ss', str(start_sec),
    '-t', str(duration_cut),
    '-i', video_path,
    '-c:v', 'libx264',
    '-c:a', 'copy',
    trimmed_path
]

result = subprocess.run(cmd, capture_output=True, text=True)

In [ ]:
if result.returncode == 0:
    print(f"✓ Обрезано: {start_sec}с - {end_sec}с")
    print(f"✓ Длительность: {duration_cut:.2f}с")
    print(f"✓ Файл сохранён: {trimmed_path}")
    print(f'   Число кадров: {total_frames}')
    print()
    # Воспроизведение обрезанного видео с embed=True
    display(Video(trimmed_path, width=600, embed=True))
else:
    print("Ошибка FFmpeg:")
    print(result.stderr)

In [ ]:
import matplotlib.pyplot as plt
import json
import os

# Путь к файлу с сохранёнными координатами ROI
roi_config_file = "roi_config.json"

# Берём первый кадр обрезанного видео
cap = cv2.VideoCapture(trimmed_path)
ret, first_frame = cap.read()
cap.release()

if ret:
    # Конвертируем BGR в RGB для matplotlib
    frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

    # Проверяем, есть ли сохранённый ROI
    if os.path.exists(roi_config_file):
        # Загружаем из файла
        with open(roi_config_file, 'r') as f:
            roi_data = json.load(f)
        x, y, w, h = roi_data['x'], roi_data['y'], roi_data['w'], roi_data['h']
        print(f"✓ ROI загружен из файла: x={x}, y={y}, w={w}, h={h}")
        print(f"  (Удалите файл '{roi_config_file}' для нового выбора)")
    else:
        # Интерактивный выбор ROI
        print("Выберите ROI (область с шариком) в открывшемся окне")
        print("Нажмите ENTER после выбора, ESC для отмены")

        roi = cv2.selectROI("Выберите ROI", first_frame, fromCenter=False, showCrosshair=True)
        cv2.destroyAllWindows()

        x, y, w, h = roi

        # Сохраняем в файл
        roi_data = {'x': int(x), 'y': int(y), 'w': int(w), 'h': int(h)}
        with open(roi_config_file, 'w') as f:
            json.dump(roi_data, f, indent=2)

        print(f"\n✓ ROI выбран и сохранён: x={x}, y={y}, w={w}, h={h}")

    # Показываем первый кадр с выделенной областью
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(frame_rgb)

    # Рисуем прямоугольник ROI
    from matplotlib.patches import Rectangle
    rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

    ax.set_title("Первый кадр с выбранным ROI")
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Ошибка: не удалось прочитать первый кадр")

В этом шаге вам нужно научиться работать с разными цветовыми пространствами и каналами изображения.

### Функция `cv2.cvtColor()`
Преобразует изображение из одного цветового пространства в другое.

**Что принимает:**
- `src` - исходное изображение (в нашем случае `roi_img`)
- `code` - код преобразования (например, `cv2.COLOR_BGR2RGB`)

**Коды преобразования:**
- `cv2.COLOR_BGR2RGB` - из BGR (формат OpenCV) в RGB (стандартный формат)
- `cv2.COLOR_BGR2HSV` - из BGR в HSV (Hue-Saturation-Value)
- `cv2.COLOR_BGR2GRAY` - из BGR в оттенки серого

### Что такое HSV?
**HSV** - это цветовая модель, которая представляет цвет через:
- **H (Hue)** - тон/оттенок цвета (0-179 в OpenCV), определяет сам цвет (красный, зеленый, синий и т.д.)
- **S (Saturation)** - насыщенность (0-255), определяет чистоту цвета (от серого до чистого цвета)
- **V (Value)** - яркость (0-255), определяет светлость цвета

HSV удобнее для выделения объектов по цвету, чем RGB.

### Функция `cv2.split()`
Разделяет многоканальное изображение на отдельные каналы.

**Что принимает:**
- Многоканальное изображение (например, RGB или HSV)

**Что возвращает:**
- Кортеж из отдельных одноканальных массивов (например, для RGB вернет 3 массива: R, G, B)


In [ ]:
# Визуализация разных цветовых пространств и каналов
import numpy as np

# Вырезаем ROI из первого кадра
roi_img = first_frame[y:y+h, x:x+w]

# TODO: Конвертируйте roi_img в RGB используя cv2.cvtColor
rgb = cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB)

# TODO: Конвертируйте roi_img в HSV
hsv = ...

# TODO: Конвертируйте roi_img в grayscale (оттенки серого)
gray = ...

# TODO: Разделите rgb на отдельные каналы R, G, B используя cv2.split
r_ch, g_ch, b_ch = cv2.split(rgb)

# TODO: Разделите hsv на отдельные каналы H, S, V
h_ch, s_ch, v_ch = ...

# Создаём полотно 2x4 (только RGB и HSV)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Строка 1: RGB оригинал, Grayscale, R, G
axes[0].imshow(rgb)
axes[0].set_title('RGB (оригинал)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(r_ch, cmap='Reds')
axes[2].set_title('R-канал (Red)', fontsize=12)
axes[2].axis('off')

axes[3].imshow(g_ch, cmap='Greens')
axes[3].set_title('G-канал (Green)', fontsize=12)
axes[3].axis('off')

# Строка 2: B, H, S, V
axes[4].imshow(b_ch, cmap='Blues')
axes[4].set_title('B-канал (Blue)', fontsize=12)
axes[4].axis('off')

axes[5].imshow(h_ch, cmap='hsv')
axes[5].set_title('H-канал (Hue/Оттенок)', fontsize=12)
axes[5].axis('off')

axes[6].imshow(s_ch, cmap='gray')
axes[6].set_title('S-канал (Saturation/Насыщенность)', fontsize=12)
axes[6].axis('off')

axes[7].imshow(v_ch, cmap='gray')
axes[7].set_title('V-канал (Value/Яркость)', fontsize=12)
axes[7].axis('off')

plt.suptitle('Анализ цветовых каналов ROI (RGB + HSV)', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
from ipywidgets import interactive, IntSlider, VBox, HBox, Label
import ipywidgets as widgets
from IPython.display import clear_output

# Функция для обновления маски
def update_mask(h_min, h_max, s_min, s_max, v_min, v_max, kernel_size):
    clear_output(wait=True)  # Очищаем предыдущий вывод

    # Создаём маску по HSV порогам
    lower = np.array([h_min, s_min, v_min])
    upper = np.array([h_max, s_max, v_max])

    mask = cv2.inRange(hsv, lower, upper)

    # Применяем морфологические операции для удаления шума и мелких точек
    if kernel_size > 0:
        # Создаём ядро для морфологических операций
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

        # Closing: сначала расширение (dilate), затем сужение (erode)
        # Убирает мелкие дырки внутри объектов
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        # Opening: сначала сужение (erode), затем расширение (dilate)
        # Убирает мелкие белые точки (шум)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # Создаём результат с СЕРЫМ фоном (чтобы черный шар был виден)
    result = np.full_like(rgb, 128)  # Серый фон (128, 128, 128)
    result[mask > 0] = rgb[mask > 0]  # Копируем пиксели там, где маска активна

    # Визуализация
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(rgb)
    axes[0].set_title('Оригинал ROI', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title(f'Маска (ядро {kernel_size}x{kernel_size})', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(result)
    axes[2].set_title('Результат (серый фон)', fontsize=12, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Создаём слайдеры
interactive_plot = interactive(
    update_mask,
    h_min=IntSlider(min=0, max=179, step=1, value=0, description='H min:'),
    h_max=IntSlider(min=0, max=179, step=1, value=179, description='H max:'),
    s_min=IntSlider(min=0, max=255, step=1, value=0, description='S min:'),
    s_max=IntSlider(min=0, max=255, step=1, value=255, description='S max:'),
    v_min=IntSlider(min=0, max=255, step=1, value=0, description='V min:'),
    v_max=IntSlider(min=0, max=255, step=1, value=100, description='V max:'),
    kernel_size=IntSlider(min=0, max=21, step=2, value=5, description='Ядро:')
)

display(interactive_plot)


В этом шаге вы научитесь создавать цветовую маску и извлекать характеристики объекта.

### Создание диапазонов для маски
Для выделения объекта по цвету нужно задать **диапазон значений** в HSV:
- `lower` - массив с минимальными значениями `[h_min, s_min, v_min]`
- `upper` - массив с максимальными значениями `[h_max, s_max, v_max]`

Используйте функцию `np.array([...])` для создания массивов.

### Функция `cv2.inRange()`
Создает бинарную маску на основе диапазона значений.

**Что принимает:**
- `src` - исходное изображение (в нашем случае `hsv`)
- `lowerb` - нижняя граница диапазона (массив `lower`)
- `upperb` - верхняя граница диапазона (массив `upper`)

**Что возвращает:**
- Бинарную маску (0 или 255), где **255** - пиксели попадающие в диапазон, **0** - остальные

### Морфологические операции
После создания маски нужно убрать шумы и заполнить дыры.

**Структурирующий элемент:**
```python
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
```
Создает матрицу эллиптической формы для морфологических операций.

Пример матрицы 5×5 (`kernel_size=5`):
```
[[0 0 1 0 0]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [0 0 1 0 0]]
```

**Операции:**
- `cv2.MORPH_CLOSE` (закрытие) - сначала расширение, потом эрозия. Заполняет **маленькие дыры** внутри объекта
- `cv2.MORPH_OPEN` (открытие) - сначала эрозия, потом расширение. Убирает **мелкий шум** снаружи объекта

### Функция `cv2.moments()`
Вычисляет моменты изображения - статистические характеристики формы.

**Что принимает:**
- Бинарное или grayscale изображение (в нашем случае `mask`)

**Что возвращает:**
- Словарь с моментами: `m00`, `m10`, `m01`, `m20`, `m11`, `m02` и т.д.

Моменты - это суммы пикселей с разными весами. Из них можно получить полезные характеристики объекта.

### Задание
Используя словарь моментов `M`, вычислите:
1. **Центр масс** - координаты `cx` и `cy` центра объекта
2. **Площадь** - количество пикселей объекта
3. **Радиус** - предполагая что объект круглый (используйте формулу площади круга: S = π·r²)


In [ ]:
# Анализ силуэта: моменты, центр масс, радиус

# Используем параметры из виджета выше (подставьте свои найденные значения)
h_min, h_max = ...
s_min, s_max = ...
v_min, v_max = ...
kernel_size = ...

# TODO: Создайте массив lower с минимальными значениями [h_min, s_min, v_min]
lower = ...

# TODO: Создайте массив upper с максимальными значениями [h_max, s_max, v_max]
upper = ...

# TODO: Создайте бинарную маску используя cv2.inRange
mask = ...

# Морфологические операции для улучшения маски
if kernel_size > 0:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

# Вычисляем моменты изображения
M = cv2.moments(mask)

if M['m00'] > 0:  # Проверяем, что есть непустая область
    # TODO: Вычислите координату X центра масс
    cx = ...
    
    # TODO: Вычислите координату Y центра масс
    cy = ...
    
    # TODO: Найдите площадь объекта
    area = ...
    
    # TODO: Вычислите радиус из площади (предполагая что объект - круг)
    # Формула площади круга: S = π·r²
    radius_from_area = ...

    
    # 2. Альтернативно: минимальная окружность
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # Берём самый большой контур
        largest_contour = max(contours, key=cv2.contourArea)
        (cx_circle, cy_circle), radius_circle = cv2.minEnclosingCircle(largest_contour)
        cx_circle, cy_circle = int(cx_circle), int(cy_circle)
        radius_circle = int(radius_circle)
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Маска
    axes[0].imshow(mask, cmap='gray')
    axes[0].plot(cx, cy, 'r+', markersize=20, markeredgewidth=3, label='Центр масс')
    axes[0].set_title('Маска + центр масс', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].axis('off')
    
    # Оригинал с наложением
    axes[1].imshow(rgb)
    # Центр масс
    axes[1].plot(cx, cy, 'r+', markersize=20, markeredgewidth=3, label=f'Центр масс ({cx}, {cy})')
    # Окружность (минимальная)
    circle = plt.Circle((cx_circle, cy_circle), radius_circle, color='lime', fill=False, linewidth=2, label=f'R={radius_circle} px')
    axes[1].add_patch(circle)
    axes[1].set_title('Результат', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Выводим результаты
    print("="*60)
    print("АНАЛИЗ СИЛУЭТА")
    print("="*60)
    print(f"Центр масс:           ({cx}, {cy})")
    print(f"Площадь:              {area:.1f} пикселей")
    print(f"Радиус (из площади):  {radius_from_area:.1f} пикселей")
    print(f"\nМинимальная окружность:")
    print(f"  Центр:              ({cx_circle}, {cy_circle})")
    print(f"  Радиус:             {radius_circle} пикселей")
    print(f"\nРазница центров:      {np.sqrt((cx - cx_circle)**2 + (cy - cy_circle)**2):.2f} пикселей")
    print("="*60)
else:
    print("⚠️ Силуэт не найден! Настройте параметры HSV выше.")

In [ ]:
# Функция для детекции центра и радиуса шара

def detect_ball(frame, roi, params):
    """
    Детекция шара через силуэт (HSV маска + моменты).

    Parameters:
    -----------
    frame : ndarray
        Полный кадр (BGR)
    roi : tuple
        (x, y, w, h) - координаты ROI
    params : dict
        Параметры: h_min, h_max, s_min, s_max, v_min, v_max, kernel_size

    Returns:
    --------
    cx, cy, radius : int or None
        Координаты центра (относительно ROI) и радиус, или (None, None, None)
    """
    x, y, w, h = roi

    # Вырезаем ROI
    roi_img = frame[y:y+h, x:x+w]

    # Конвертируем в HSV
    hsv = cv2.cvtColor(roi_img, cv2.COLOR_BGR2HSV)

    # Создаём маску
    lower = np.array([params['h_min'], params['s_min'], params['v_min']])
    upper = np.array([params['h_max'], params['s_max'], params['v_max']])
    mask = cv2.inRange(hsv, lower, upper)

    # Морфологические операции
    kernel_size = params.get('kernel_size', 5)
    if kernel_size > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # Поиск контуров и минимальная окружность
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Берём самый большой контур
        largest_contour = max(contours, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(largest_contour)
        return int(cx), int(cy), int(radius)

    return None, None, None

print("✓ Функция детекции загружена!")

In [ ]:
# Обработка видео: трекинг шара + графики
import pandas as pd
from tqdm.notebook import tqdm

# ===== НАСТРОЙКИ =====
# Параметры для детекции (силуэт)
params = {
    'h_min': 0, 'h_max': 179,
    's_min': 0, 's_max': 255,
    'v_min': 0, 'v_max': 112,
    'kernel_size': 21
}

# Путь для сохранения
output_video_path = 'ball_tracked.mp4'

# ===== ОБРАБОТКА ВИДЕО =====
print("Метод детекции: силуэт (HSV)")
print(f"Обработка видео: {trimmed_path}")
print(f"ROI: x={x}, y={y}, w={w}, h={h}")

cap = cv2.VideoCapture(trimmed_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps:.2f}, Кадров: {total_frames}, Длительность: {total_frames/fps:.2f} сек")

# Создаем writer для видео
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Для сохранения результатов
results = []

# Обработка кадров
for frame_num in tqdm(range(total_frames), desc='Обработка кадров'):
    ret, frame = cap.read()
    if not ret:
        break
    
    # Детекция
    cx, cy, radius = detect_ball(frame, (x, y, w, h), params)
    
    if cx is not None:
        # Координаты в полном кадре
        cx_full = cx + x
        cy_full = cy + y
        
        # Рисуем окружность и центр
        cv2.circle(frame, (cx_full, cy_full), radius, (0, 255, 0), 2)
        cv2.drawMarker(frame, (cx_full, cy_full), (0, 0, 255), 
                      cv2.MARKER_CROSS, 20, 2)
        
        # Текст с информацией
        cv2.putText(frame, f'Center: ({cx_full}, {cy_full})', 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f'Radius: {radius} px', 
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Сохраняем результаты
        time = frame_num / fps
        results.append({
            'frame': frame_num,
            'time': time,
            'x': cx_full,
            'y': cy_full,
            'radius': radius
        })
    
    # Рисуем рамку ROI
    cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
    
    # Записываем кадр
    out.write(frame)

cap.release()
out.release()

print(f"\n✓ Обработано кадров: {len(results)} / {total_frames}")
print(f"✓ Видео сохранено: {output_video_path}")

# Сохраняем результаты в CSV
df = pd.DataFrame(results)
csv_path = 'tracking_data.csv'
df.to_csv(csv_path, index=False)
print(f"✓ Данные сохранены: {csv_path}")


In [ ]:
def analyze_tracking_results():
    """Анализ и визуализация результатов трекинга."""
    import os

    # Читаем данные из CSV
    csv_path = 'tracking_data.csv'

    if not os.path.exists(csv_path):
        print(f"⚠️ Файл {csv_path} не найден!")
        print("   Сначала запустите обработку видео")
        return

    df = pd.read_csv(csv_path)

    if df.empty:
        print("⚠️ CSV файл пустой!")
        return

    # Путь к видео
    output_video_path = 'ball_tracked.mp4'

    # Графики
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    axes[0].plot(df['time'], df['x'], 'b-', linewidth=1.5)
    axes[0].set_ylabel('X (пиксели)', fontsize=12)
    axes[0].set_title('Координаты центра шара', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df['time'], df['y'], 'r-', linewidth=1.5)
    axes[1].set_ylabel('Y (пиксели)', fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].invert_yaxis()  # Инвертируем Y (в изображениях Y растёт вниз)

    axes[2].plot(df['time'], df['radius'], 'g-', linewidth=1.5)
    axes[2].set_ylabel('Радиус (пиксели)', fontsize=12)
    axes[2].set_xlabel('Время (секунды)', fontsize=12)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Статистика
    print("\n" + "="*60)
    print("СТАТИСТИКА")
    print("="*60)
    print(f"Кадров обработано:   {len(df)}")
    print(f"Средний радиус:      {df['radius'].mean():.2f} ± {df['radius'].std():.2f} пикселей")
    print(f"Диапазон X:          {df['x'].min():.0f} - {df['x'].max():.0f} пикселей")
    print(f"Диапазон Y:          {df['y'].min():.0f} - {df['y'].max():.0f} пикселей")
    print("="*60)

    # Конвертируем в браузер-совместимый формат (H.264)
    output_video_web = 'ball_tracked_web.mp4'

    if os.path.exists(output_video_web):
        print(f"\n✓ Конвертированное видео найдено: {output_video_web}")
    elif os.path.exists(output_video_path):
        print(f"\n🔄 Конвертация в H.264 для браузера...")

        convert_cmd = [
            'ffmpeg', '-y',
            '-i', output_video_path,
            '-c:v', 'libx264',
            '-preset', 'fast',
            '-crf', '23',
            '-c:a', 'copy',
            output_video_web
        ]

        convert_result = subprocess.run(convert_cmd, capture_output=True, text=True)

        if convert_result.returncode == 0:
            print(f"✓ Конвертация завершена: {output_video_web}")
        else:
            print("⚠️ Ошибка конвертации FFmpeg:")
            print(convert_result.stderr)
    else:
        print(f"⚠️ Видео файл не найден: {output_video_path}")
        print("   Сначала запустите обработку видео")
        return

    # Показываем видео
    print("\n📹 Обработанное видео:")
    if os.path.exists(output_video_web):
        display(Video(output_video_web, width=800, embed=True))
    else:
        display(Video(output_video_path, width=800, embed=True))

In [ ]:
analyze_tracking_results()

In [ ]:
# Загружаем данные трекинга
data = pd.read_csv('tracking_data.csv')
data

In [ ]:
x = data['x'].to_numpy()
y = data['y'].to_numpy()
plt.plot(x, -y, 'o')

In [ ]:
# Фитинг центра вращения (подвеса) и длины маятника L
from scipy.optimize import least_squares
import numpy as np

def fit_circle(x, y):
    """
    Фитинг окружности к точкам (x, y) методом наименьших квадратов.
    
    Returns:
    --------
    x_center, y_center, radius : float
        Параметры окружности
    """
    # Начальное приближение: центр = среднее, радиус = std
    x_mean, y_mean = x.mean(), y.mean()
    radius_init = np.sqrt(((x - x_mean)**2 + (y - y_mean)**2).mean())
    
    # Функция остатков: расстояние от точки до окружности
    def residuals(params):
        xc, yc, r = params
        distances = np.sqrt((x - xc)**2 + (y - yc)**2)
        return distances - r
    
    # Минимизация
    params_init = [x_mean, y_mean, radius_init]
    result = least_squares(residuals, params_init)
    
    x_center, y_center, radius = result.x
    return x_center, y_center, radius


# Загружаем данные (используем метод silhouette)
method = 'silhouette'
csv_path = f'tracking_data_{method}.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    # Координаты центра шарика
    x = df['x'].to_numpy()
    y = df['y'].to_numpy()
    
    # Фитинг окружности
    print("Фитинг окружности к траектории маятника...")
    x_pivot, y_pivot, L = fit_circle(x, y)
    
    # Визуализация
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    
    # Траектория (инвертируем Y для правильной ориентации)
    ax.plot(x, y, 'o', markersize=3, alpha=0.5, label='Траектория шарика')
    
    # Точка подвеса
    ax.plot(x_pivot, y_pivot, 'r*', markersize=20, label=f'Точка подвеса ({x_pivot:.1f}, {y_pivot:.1f})')
    
    # Окружность (fitted)
    theta = np.linspace(0, 2*np.pi, 1000)
    x_circle = x_pivot + L * np.cos(theta)
    y_circle = y_pivot + L * np.sin(theta)
    ax.plot(x_circle, y_circle, 'r--', linewidth=2, alpha=0.7, label=f'Окружность (R={L:.1f} px)')
    
    # Линия от подвеса до нескольких точек траектории
    for i in range(0, len(x), len(x)//10):  # Каждая 10-я точка
        ax.plot([x_pivot, x[i]], [y_pivot, y[i]], 'gray', linewidth=0.5, alpha=0.3)
    
    ax.set_xlabel('X (пиксели)', fontsize=12)
    ax.set_ylabel('Y (пиксели)', fontsize=12)
    ax.set_title('Фитинг маятника: центр вращения и длина L', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.1)
    ax.set_aspect('equal', adjustable='box')
    ax.invert_yaxis()  # Инвертируем Y (в изображениях Y растёт вниз)
    
    plt.tight_layout()
    plt.show()
    
    # Результаты
    print("" + "="*60)
    print("РЕЗУЛЬТАТЫ ФИТИНГА МАЯТНИКА")
    print("="*60)
    print(f"Центр вращения (подвес):")
    print(f"  X = {x_pivot:.2f} пикселей")
    print(f"  Y = {y_pivot:.2f} пикселей")
    print(f"Длина маятника:")
    print(f"  L = {L:.2f} пикселей")
    
    # Оценка качества фитинга
    distances = np.sqrt((x - x_pivot)**2 + (y - y_pivot)**2)
    residuals = np.abs(distances - L)
    print(f"Качество фитинга:")
    print(f"  Средняя ошибка: {residuals.mean():.3f} пикселей")
    print(f"  Макс. ошибка:   {residuals.max():.3f} пикселей")
    print(f"  Std ошибки:     {residuals.std():.3f} пикселей")
    print("="*60)
    
else:
    print(f"⚠️ Файл {csv_path} не найден!")
    print(f"   Запустите ячейку 12 с method='{method}' для создания данных")


In [ ]:
# Расчет масштаба и погрешностей

import numpy as np

# Известные данные
ball_diameter_real = 49.71  # мм (реальный диаметр шарика)
ball_diameter_error = 4.0   # мм (погрешность измерения диаметра)

ball_diameter_real_m = ball_diameter_real / 1000  # м
ball_radius_real_m = ball_diameter_real_m / 2  # радиус в метрах
ball_radius_error_m = (ball_diameter_error / 2) / 1000  # погрешность радиуса в метрах

# Загружаем данные трекинга
method = 'silhouette'
csv_path = f'tracking_data_{method}.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    # Средний радиус шарика в пикселях
    ball_radius_px = df['radius'].mean()
    ball_radius_std_px = df['radius'].std()  # дисперсия радиуса в пикселях
    
    # ОБРАТНЫЙ масштаб: пиксели/метр (более наглядно!)
    scale_px_per_m = ball_radius_px / ball_radius_real_m  # пикселей/метр
    
    # Погрешность масштаба (относительная)
    # δscale/scale = sqrt((δr_real/r_real)^2 + (δr_px/r_px)^2)
    relative_error_scale = np.sqrt(
        (ball_radius_error_m / ball_radius_real_m)**2 + 
        (ball_radius_std_px / ball_radius_px)**2
    )
    scale_error = scale_px_per_m * relative_error_scale
    
    # Преобразуем длину маятника L из пикселей в метры
    if 'L' in globals() and 'x_pivot' in globals():
        # L в метрах = L_px / scale_px_per_m
        L_meters = L / scale_px_per_m
        L_mm = L_meters * 1000
        
        # Погрешность L_px из фитинга (используем std остатков как оценку)
        distances = np.sqrt((df['x'].to_numpy() - x_pivot)**2 + 
                           (df['y'].to_numpy() - y_pivot)**2)
        L_px_error = np.std(distances - L)
        
        # Полная погрешность L в метрах
        # δL/L = sqrt((δL_px/L_px)^2 + (δscale/scale)^2)
        relative_error_L = np.sqrt(
            (L_px_error / L)**2 + 
            relative_error_scale**2
        )
        L_error_m = L_meters * relative_error_L
        L_error_mm = L_error_m * 1000
        
    else:
        print("Внимание: переменные L, x_pivot, y_pivot не найдены!")
        print("Запустите сначала ячейку с фитингом маятника")
        L_meters = None
        L_mm = None
        L_error_m = None
        L_error_mm = None
    
    # Результаты
    print("="*70)
    print("РАСЧЕТ МАСШТАБА И ПОГРЕШНОСТЕЙ")
    print("="*70)
    print(f"Реальный размер шарика:")
    print(f"  Диаметр:  {ball_diameter_real:.2f} +/- {ball_diameter_error:.2f} мм")
    print(f"  Радиус:   {ball_radius_real_m*1000:.2f} +/- {ball_radius_error_m*1000:.2f} мм")
    
    print(f"Размер шарика в пикселях (из трекинга):")
    print(f"  Средний радиус: {ball_radius_px:.2f} +/- {ball_radius_std_px:.2f} пикселей")
    
    print(f"Масштаб (обратный - пиксели на метр):")
    print(f"  {scale_px_per_m:.2f} +/- {scale_error:.2f} пикселей/метр")
    print(f"  {scale_px_per_m/1000:.2f} +/- {scale_error/1000:.2f} пикселей/мм")
    print(f"  Относительная погрешность: {relative_error_scale*100:.2f}%")
    
    if L_meters is not None:
        print(f"Длина маятника L:")
        print(f"  Пиксели: {L:.2f} +/- {L_px_error:.2f} px")
        print(f"  Метры:   {L_meters:.4f} +/- {L_error_m:.4f} м")
        print(f"  Мм:      {L_mm:.2f} +/- {L_error_mm:.2f} мм")
        print(f"  Относительная погрешность: {relative_error_L*100:.2f}%")
    
    print("="*70)
    print("Вклад в погрешность L:")
    if L_meters is not None:
        contrib_L_px = (L_px_error / L)**2
        contrib_scale = relative_error_scale**2
        total_variance = contrib_L_px + contrib_scale
        
        print(f"  От погрешности фитинга L_px: {np.sqrt(contrib_L_px/total_variance)*100:.1f}%")
        print(f"  От погрешности масштаба: {np.sqrt(contrib_scale/total_variance)*100:.1f}%")
    print("="*70)
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # График 1: Радиус шарика в пикселях
    axes[0].plot(df['time'], df['radius'], 'b-', linewidth=1, alpha=0.7)
    axes[0].axhline(ball_radius_px, color='r', linestyle='--', linewidth=2, 
                   label=f'Средний: {ball_radius_px:.2f} px')
    axes[0].fill_between(df['time'], 
                        ball_radius_px - ball_radius_std_px, 
                        ball_radius_px + ball_radius_std_px,
                        alpha=0.2, color='red', label=f'+/-sigma: {ball_radius_std_px:.2f} px')
    axes[0].set_xlabel('Время (секунды)', fontsize=12)
    axes[0].set_ylabel('Радиус (пиксели)', fontsize=12)
    axes[0].set_title('Радиус шарика в пикселях', fontsize=13, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # График 2: Радиус шарика в мм
    axes[1].plot(df['time'], df['radius'] / (scale_px_per_m/1000), 'g-', linewidth=1, alpha=0.7)
    axes[1].axhline(ball_radius_real_m * 1000, color='r', linestyle='--', linewidth=2,
                   label=f'Реальный: {ball_radius_real_m*1000:.2f} мм')
    axes[1].fill_between([df['time'].min(), df['time'].max()],
                        (ball_radius_real_m - ball_radius_error_m) * 1000,
                        (ball_radius_real_m + ball_radius_error_m) * 1000,
                        alpha=0.2, color='red', label=f'+/- {ball_radius_error_m*1000:.2f} мм')
    axes[1].set_xlabel('Время (секунды)', fontsize=12)
    axes[1].set_ylabel('Радиус (мм)', fontsize=12)
    axes[1].set_title('Радиус шарика в миллиметрах', fontsize=13, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Сохраняем переменные
    print(f"Переменные сохранены:")
    print(f"  scale_px_per_m = {scale_px_per_m:.2f} +/- {scale_error:.2f} пикселей/метр")
    
else:
    print(f"Файл {csv_path} не найден!")
    print(f"Запустите ячейку 12 с method='{method}' для создания данных")


In [ ]:
# Разложение в ряд Фурье (FFT анализ)

import numpy as np
from scipy.fft import fft, fftfreq

# Загружаем данные
method = 'silhouette'
csv_path = f'tracking_data_{method}.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    # Координаты и время
    t = df['time'].to_numpy()
    x = df['x'].to_numpy()
    y = df['y'].to_numpy()
    
    # Параметры
    N = len(t)  # количество точек
    dt = np.mean(np.diff(t))  # средний шаг по времени
    fps = 1 / dt  # частота дискретизации
    
    # Центрируем данные (вычитаем среднее)
    x_centered = x - x.mean()
    y_centered = y - y.mean()
    
    # FFT для x и y
    fft_x = fft(x_centered)
    fft_y = fft(y_centered)
    
    # Частоты
    freqs = fftfreq(N, dt)
    
    # Берём только положительные частоты (спектр симметричен)
    positive_freqs = freqs[:N//2]
    fft_x_positive = np.abs(fft_x[:N//2]) * 2 / N  # амплитуды
    fft_y_positive = np.abs(fft_y[:N//2]) * 2 / N
    
    # Находим пики (основные гармоники)
    n_peaks = 5  # количество пиков для отображения
    
    # Для x
    peaks_x_idx = np.argsort(fft_x_positive)[-n_peaks:][::-1]
    peaks_x_freq = positive_freqs[peaks_x_idx]
    peaks_x_amp = fft_x_positive[peaks_x_idx]
    
    # Для y
    peaks_y_idx = np.argsort(fft_y_positive)[-n_peaks:][::-1]
    peaks_y_freq = positive_freqs[peaks_y_idx]
    peaks_y_amp = fft_y_positive[peaks_y_idx]
    
    # Основная частота (самая мощная гармоника)
    main_freq_x = peaks_x_freq[0]
    main_freq_y = peaks_y_freq[0]
    
    # Период колебаний
    period_x = 1 / main_freq_x if main_freq_x > 0 else 0
    period_y = 1 / main_freq_y if main_freq_y > 0 else 0
    
    # Результаты
    print("="*70)
    print("АНАЛИЗ ФУРЬЕ")
    print("="*70)
    print(f"Параметры сигнала:")
    print(f"  Количество точек: {N}")
    print(f"  Длительность: {t[-1] - t[0]:.2f} сек")
    print(f"  Частота дискретизации: {fps:.2f} Гц")
    print(f"  Разрешение по частоте: {freqs[1]:.4f} Гц")
    
    print(f"Основная частота:")
    print(f"  X: {main_freq_x:.4f} Гц (период {period_x:.3f} сек)")
    print(f"  Y: {main_freq_y:.4f} Гц (период {period_y:.3f} сек)")
    
    print(f"Топ {n_peaks} гармоник для X:")
    for i, (freq, amp) in enumerate(zip(peaks_x_freq, peaks_x_amp)):
        print(f"  {i+1}. f = {freq:.4f} Гц, A = {amp:.2f} пикселей")
    
    print(f"Топ {n_peaks} гармоник для Y:")
    for i, (freq, amp) in enumerate(zip(peaks_y_freq, peaks_y_amp)):
        print(f"  {i+1}. f = {freq:.4f} Гц, A = {amp:.2f} пикселей")
    
    print("="*70)
    
    # Визуализация
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # График 1: Исходные сигналы X и Y
    ax = axes[0, 0]
    ax.plot(t, x_centered, 'b-', linewidth=1, alpha=0.7, label='X (центрированный)')
    ax.set_xlabel('Время (секунды)', fontsize=11)
    ax.set_ylabel('X - X_mean (пиксели)', fontsize=11)
    ax.set_title('Колебания по X', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    ax = axes[0, 1]
    ax.plot(t, y_centered, 'r-', linewidth=1, alpha=0.7, label='Y (центрированный)')
    ax.set_xlabel('Время (секунды)', fontsize=11)
    ax.set_ylabel('Y - Y_mean (пиксели)', fontsize=11)
    ax.set_title('Колебания по Y', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # График 2: Спектр X
    ax = axes[1, 0]
    ax.plot(positive_freqs, fft_x_positive, 'b-', linewidth=1, alpha=0.7)
    # Отмечаем пики
    ax.plot(peaks_x_freq, peaks_x_amp, 'ro', markersize=8, label='Пики')
    for i, (freq, amp) in enumerate(zip(peaks_x_freq[:3], peaks_x_amp[:3])):
        ax.annotate(f'{freq:.3f} Гц', 
                   xy=(freq, amp), 
                   xytext=(10, 10), 
                   textcoords='offset points',
                   fontsize=9,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
    ax.set_xlabel('Частота (Гц)', fontsize=11)
    ax.set_ylabel('Амплитуда (пиксели)', fontsize=11)
    ax.set_title('Спектр Фурье для X', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 2)  # Ограничиваем диапазон для лучшей видимости
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # График 3: Спектр Y
    ax = axes[1, 1]
    ax.plot(positive_freqs, fft_y_positive, 'r-', linewidth=1, alpha=0.7)
    # Отмечаем пики
    ax.plot(peaks_y_freq, peaks_y_amp, 'bo', markersize=8, label='Пики')
    for i, (freq, amp) in enumerate(zip(peaks_y_freq[:3], peaks_y_amp[:3])):
        ax.annotate(f'{freq:.3f} Гц', 
                   xy=(freq, amp), 
                   xytext=(10, 10), 
                   textcoords='offset points',
                   fontsize=9,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
    ax.set_xlabel('Частота (Гц)', fontsize=11)
    ax.set_ylabel('Амплитуда (пиксели)', fontsize=11)
    ax.set_title('Спектр Фурье для Y', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 2)  # Ограничиваем диапазон для лучшей видимости
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Дополнительный график: спектр в логарифмическом масштабе
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # X спектр (лог)
    ax = axes[0]
    ax.semilogy(positive_freqs, fft_x_positive, 'b-', linewidth=1, alpha=0.7)
    ax.plot(peaks_x_freq, peaks_x_amp, 'ro', markersize=8, label='Пики')
    ax.set_xlabel('Частота (Гц)', fontsize=11)
    ax.set_ylabel('Амплитуда (лог. шкала)', fontsize=11)
    ax.set_title('Спектр X (логарифмическая шкала)', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 5)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend()
    
    # Y спектр (лог)
    ax = axes[1]
    ax.semilogy(positive_freqs, fft_y_positive, 'r-', linewidth=1, alpha=0.7)
    ax.plot(peaks_y_freq, peaks_y_amp, 'bo', markersize=8, label='Пики')
    ax.set_xlabel('Частота (Гц)', fontsize=11)
    ax.set_ylabel('Амплитуда (лог. шкала)', fontsize=11)
    ax.set_title('Спектр Y (логарифмическая шкала)', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 5)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Сохраняем переменные
    print(f"Переменные сохранены: main_freq_x={main_freq_x:.4f} Гц, main_freq_y={main_freq_y:.4f} Гц")
    
else:
    print(f"Файл {csv_path} не найден!")
    print(f"Запустите ячейку 12 с method='{method}' для создания данных")


In [ ]:
t = data['time']

In [ ]:
plt.figure(figsize=(10, 5))
n = 500
plt.plot(y[:n])
plt.plot(203 + 2.78*np.sin(2*np.pi*1.215*t[:n]+8) + 1.48*np.sin(2*np.pi*0.516*t[:n]))

In [ ]:
# Фитинг затухающих колебаний: X (1 частота) + Y (3 частоты)

import numpy as np
from scipy.optimize import curve_fit

# Загружаем данные
method = 'silhouette'
csv_path = f'tracking_data_{method}.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    t = df['time'].to_numpy()
    x = df['x'].to_numpy()
    y = df['y'].to_numpy()
    
    # Модель для X: одна частота с затуханием
    def model_x(t, x0, A, gamma, omega, phi):
        return x0 + A * np.exp(-gamma * t) * np.sin(omega * t + phi)
    
    # Модель для Y: ТРИ частоты с общим затуханием
    def model_y_3freq(t, y0, A1, omega1, phi1, A2, omega2, phi2, A3, omega3, phi3, gamma):
        decay = np.exp(-gamma * t)
        return y0 + decay * (
            A1 * np.sin(omega1 * t + phi1) + 
            A2 * np.sin(omega2 * t + phi2) +
            A3 * np.sin(omega3 * t + phi3)
        )
    
    # Начальные приближения для X
    x0_init = x.mean()
    A_x_init = (x.max() - x.min()) / 2
    omega_x_init = 2 * np.pi * 0.60
    gamma_init = 0.01
    p0_x = [x0_init, A_x_init, gamma_init, omega_x_init, 0]
    
    # Начальные приближения для Y (3 частоты из FFT)
    y0_init = y.mean()
    p0_y = [
        y0_init,           # y0
        2.78,              # A1
        2*np.pi*1.22,      # omega1 (основная ~1.2 Гц)
        1.5,               # phi1
        1.48,              # A2
        2*np.pi*0.616,     # omega2 (~0.6 Гц)
        0,                 # phi2
        1.41,              # A3 (третья частота!)
        2*np.pi*0.599,     # omega3 (~0.6 Гц, близкая ко второй)
        0,                 # phi3
        0.01               # gamma
    ]
    
    # Фитинг
    print('='*70)
    print('ФИТИНГ ЗАТУХАЮЩИХ КОЛЕБАНИЙ (Y с 3 частотами)')
    print('='*70)
    
    popt_x = None
    popt_y = None
    
    # Фитинг X
    try:
        popt_x, pcov_x = curve_fit(model_x, t, x, p0=p0_x, maxfev=10000)
        x_fitted = model_x(t, *popt_x)
        perr_x = np.sqrt(np.diag(pcov_x))
        residual_x = x - x_fitted
        rms_x = np.sqrt(np.mean(residual_x**2))
        r_squared_x = 1 - np.sum(residual_x**2) / np.sum((x - x.mean())**2)
        
        x0_fit, A_x_fit, gamma_x_fit, omega_x_fit, phi_x_fit = popt_x
        f_x_fit = omega_x_fit / (2 * np.pi)
        T_x_fit = 1 / f_x_fit
        
        print(f'X: успешно! R² = {r_squared_x:.5f}')
    except Exception as e:
        print(f'X: ошибка фитинга - {e}')
    
    # Фитинг Y (3 частоты)
    try:
        popt_y, pcov_y = curve_fit(model_y_3freq, t, y, p0=p0_y, maxfev=20000)
        y_fitted = model_y_3freq(t, *popt_y)
        perr_y = np.sqrt(np.diag(pcov_y))
        residual_y = y - y_fitted
        rms_y = np.sqrt(np.mean(residual_y**2))
        r_squared_y = 1 - np.sum(residual_y**2) / np.sum((y - y.mean())**2)
        
        y0_fit, A_y1_fit, omega_y1_fit, phi_y1_fit, A_y2_fit, omega_y2_fit, phi_y2_fit, A_y3_fit, omega_y3_fit, phi_y3_fit, gamma_y_fit = popt_y
        f_y1_fit = omega_y1_fit / (2 * np.pi)
        f_y2_fit = omega_y2_fit / (2 * np.pi)
        f_y3_fit = omega_y3_fit / (2 * np.pi)
        
        print(f'Y: успешно! R² = {r_squared_y:.5f}')
    except Exception as e:
        print(f'Y: ошибка фитинга - {e}')
    
    # Результаты X
    if popt_x is not None:
        print(f"РЕЗУЛЬТАТЫ X:")
        print(f"  Частота f = {f_x_fit:.5f} Гц")
        print(f"  Период T = {T_x_fit:.5f} сек")
        print(f"  Затухание γ = {gamma_x_fit:.5f} с⁻¹")
        print(f"  Амплитуда A = {A_x_fit:.2f} пикс")
        print(f"  Фаза φ = {phi_x_fit:.4f} рад ({np.degrees(phi_x_fit):.1f}°)")
        print(f"  RMS остатка = {rms_x:.3f} пикс")
    
    # Результаты Y
    if popt_y is not None:
        print(f"РЕЗУЛЬТАТЫ Y (3 частоты):")
        print(f"  Частота 1: f1 = {f_y1_fit:.5f} Гц (A1 = {A_y1_fit:.3f} пикс, φ1 = {np.degrees(phi_y1_fit):.1f}°)")
        print(f"  Частота 2: f2 = {f_y2_fit:.5f} Гц (A2 = {A_y2_fit:.3f} пикс, φ2 = {np.degrees(phi_y2_fit):.1f}°)")
        print(f"  Частота 3: f3 = {f_y3_fit:.5f} Гц (A3 = {A_y3_fit:.3f} пикс, φ3 = {np.degrees(phi_y3_fit):.1f}°)")
        print(f"  Затухание γ = {gamma_y_fit:.5f} с⁻¹")
        print(f"  Соотношение f1/f2 = {f_y1_fit/f_y2_fit:.3f}")
        print(f"  Соотношение f1/f3 = {f_y1_fit/f_y3_fit:.3f}")
        print(f"  RMS остатка = {rms_y:.3f} пикс")
    
    # Сравнение
    if popt_x is not None and popt_y is not None:
        print(f"СРАВНЕНИЕ:")
        print(f"  Затухание X: γ_x = {gamma_x_fit:.5f} с⁻¹")
        print(f"  Затухание Y: γ_y = {gamma_y_fit:.5f} с⁻¹")
        print(f"  Разница: {abs(gamma_x_fit - gamma_y_fit):.6f} с⁻¹")
        print(f"  Улучшение R² для Y: было ~0.62, стало {r_squared_y:.5f}")
    
    print('='*70)
    print('Переменные сохранены: popt_x, popt_y, f_x_fit, T_x_fit')
else:
    print('Файл не найден')


In [ ]:
# Детальное сравнение данных и фита (первые 500 кадров)

import numpy as np
import matplotlib.pyplot as plt

# Проверяем что фитинг был выполнен
if 'popt_x' not in globals() or 'popt_y' not in globals():
    print('Сначала запустите ячейку с фитингом!')
else:
    # Модели
    def model_x(t, x0, A, gamma, omega, phi):
        return x0 + A * np.exp(-gamma * t) * np.sin(omega * t + phi)
    
    # Модель Y с 3 частотами (обновленная!)
    def model_y_3freq(t, y0, A1, omega1, phi1, A2, omega2, phi2, A3, omega3, phi3, gamma):
        decay = np.exp(-gamma * t)
        return y0 + decay * (
            A1 * np.sin(omega1 * t + phi1) + 
            A2 * np.sin(omega2 * t + phi2) +
            A3 * np.sin(omega3 * t + phi3)
        )
    
    # Загружаем данные
    method = 'silhouette'
    csv_path = f'tracking_data_{method}.csv'
    
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        
        # Первые 500 кадров
        n = min(500, len(df))
        t_zoom = df['time'].to_numpy()[:n]
        x_zoom = df['x'].to_numpy()[:n]
        y_zoom = df['y'].to_numpy()[:n]
        
        # Вычисляем фит
        if popt_x is not None:
            x_fit_zoom = model_x(t_zoom, *popt_x)
            residual_x_zoom = x_zoom - x_fit_zoom
        
        if popt_y is not None:
            y_fit_zoom = model_y_3freq(t_zoom, *popt_y)
            residual_y_zoom = y_zoom - y_fit_zoom
        
        # Визуализация
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
        
        # X: данные vs фит
        ax = axes[0, 0]
        ax.plot(t_zoom, x_zoom, 'b.', markersize=4, alpha=0.6, label='Данные')
        if popt_x is not None:
            ax.plot(t_zoom, x_fit_zoom, 'r-', linewidth=2, label='Фит')
        ax.set_ylabel('X (пиксели)', fontsize=11)
        ax.set_title(f'X: Данные vs Фит (первые {n} кадров)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Y: данные vs фит
        ax = axes[0, 1]
        ax.plot(t_zoom, y_zoom, 'b.', markersize=4, alpha=0.6, label='Данные')
        if popt_y is not None:
            ax.plot(t_zoom, y_fit_zoom, 'r-', linewidth=2, label='Фит (3 частоты)')
        ax.set_ylabel('Y (пиксели)', fontsize=11)
        ax.set_title(f'Y: Данные vs Фит (первые {n} кадров)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # X: остаток
        ax = axes[1, 0]
        if popt_x is not None:
            ax.plot(t_zoom, residual_x_zoom, 'gray', linewidth=1)
            ax.fill_between(t_zoom, residual_x_zoom, alpha=0.3, color='gray')
            ax.axhline(0, color='black', linestyle='--', linewidth=1)
            rms_x_zoom = np.sqrt(np.mean(residual_x_zoom**2))
            ax.set_title(f'X: Остаток (RMS={rms_x_zoom:.3f} пикс)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Остаток (пиксели)', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # Y: остаток
        ax = axes[1, 1]
        if popt_y is not None:
            ax.plot(t_zoom, residual_y_zoom, 'gray', linewidth=1)
            ax.fill_between(t_zoom, residual_y_zoom, alpha=0.3, color='gray')
            ax.axhline(0, color='black', linestyle='--', linewidth=1)
            rms_y_zoom = np.sqrt(np.mean(residual_y_zoom**2))
            ax.set_title(f'Y: Остаток (RMS={rms_y_zoom:.3f} пикс)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Остаток (пиксели)', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # X: зум на первые 5 периодов
        ax = axes[2, 0]
        t_detail = t_zoom[:150]
        x_detail = x_zoom[:150]
        if popt_x is not None:
            x_fit_detail = x_fit_zoom[:150]
            ax.plot(t_detail, x_detail, 'bo', markersize=5, alpha=0.6, label='Данные')
            ax.plot(t_detail, x_fit_detail, 'r-', linewidth=2, label='Фит')
        else:
            ax.plot(t_detail, x_detail, 'bo', markersize=5, alpha=0.6, label='Данные')
        ax.set_xlabel('Время (секунды)', fontsize=11)
        ax.set_ylabel('X (пиксели)', fontsize=11)
        ax.set_title('X: Детали (первые ~5 сек)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Y: зум на первые 5 периодов
        ax = axes[2, 1]
        y_detail = y_zoom[:150]
        if popt_y is not None:
            y_fit_detail = y_fit_zoom[:150]
            ax.plot(t_detail, y_detail, 'bo', markersize=5, alpha=0.6, label='Данные')
            ax.plot(t_detail, y_fit_detail, 'r-', linewidth=2, label='Фит (3 частоты)')
        else:
            ax.plot(t_detail, y_detail, 'bo', markersize=5, alpha=0.6, label='Данные')
        ax.set_xlabel('Время (секунды)', fontsize=11)
        ax.set_ylabel('Y (пиксели)', fontsize=11)
        ax.set_title('Y: Детали (первые ~5 сек)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Статистика
        print('='*70)
        print(f'ДЕТАЛЬНЫЙ АНАЛИЗ (первые {n} кадров)')
        print('='*70)
        if popt_x is not None:
            print(f'X: RMS остатка = {rms_x_zoom:.3f} пикселей')
            print(f'   Макс. ошибка = {np.max(np.abs(residual_x_zoom)):.3f} пикселей')
        if popt_y is not None:
            print(f'Y: RMS остатка = {rms_y_zoom:.3f} пикселей (модель с 3 частотами)')
            print(f'   Макс. ошибка = {np.max(np.abs(residual_y_zoom)):.3f} пикселей')
        print('='*70)
    else:
        print('Файл данных не найден!')


In [ ]:
# Финальный расчет g (ускорение свободного падения)

import numpy as np

print('='*70)
print('РАСЧЕТ УСКОРЕНИЯ СВОБОДНОГО ПАДЕНИЯ g')
print('='*70)

# Проверяем наличие необходимых переменных
required_vars = {
    'T_x_fit': 'Период из фитинга X',
    'L': 'Длина маятника (пиксели)',
    'scale_px_per_m': 'Масштаб',
    'x_pivot': 'Центр вращения X',
    'y_pivot': 'Центр вращения Y'
}

missing = []
for var, desc in required_vars.items():
    if var not in globals():
        missing.append(f'{var} ({desc})')

if missing:
    print('Ошибка: не найдены переменные:')
    for m in missing:
        print(f'  - {m}')
    print('Запустите предыдущие ячейки:')
    print('  1. Фитинг маятника (центр вращения и L)')
    print('  2. Расчет масштаба')
    print('  3. Фитинг затухающих колебаний')
else:
    # ===== ИСХОДНЫЕ ДАННЫЕ =====
    
    # Период (из фитинга X)
    T = T_x_fit  # секунды
    
    # Погрешность периода из фитинга
    if 'perr_x' in globals() and perr_x is not None:
        # perr_x[3] - погрешность omega, omega = 2*pi*f = 2*pi/T
        # delta_T/T = delta_omega/omega
        delta_T = T * (perr_x[3] / popt_x[3]) if popt_x[3] != 0 else 0.001
    else:
        # Оценка из разрешения FFT
        delta_T = 0.001  # ~1 мс (консервативная оценка)
    
    # Длина маятника в метрах
    L_px = L  # пиксели (из фитинга окружности)
    L_m = L_px / scale_px_per_m  # метры
    
    # Погрешность длины
    # Из фитинга окружности
    if 'residuals' in dir():
        L_px_error = np.std(residuals)
    else:
        # Оценка: погрешность ~1-2 пикселя
        L_px_error = 2.0
    
    # Погрешность масштаба (из ячейки с масштабом)
    if 'relative_error_scale' in globals():
        rel_error_scale = relative_error_scale
    else:
        # Оценка из погрешностей измерений
        ball_radius_error_m = 0.001  # 1 мм (половина от 2 мм)
        ball_radius_real_m = 0.04729 / 2  # из диаметра 47.29 мм
        ball_radius_std_px = 0.5  # примерная оценка
        ball_radius_px = 26  # примерно
        rel_error_scale = np.sqrt(
            (ball_radius_error_m / ball_radius_real_m)**2 + 
            (ball_radius_std_px / ball_radius_px)**2
        )
    
    # Полная погрешность L
    rel_error_L = np.sqrt(
        (L_px_error / L_px)**2 + 
        rel_error_scale**2
    )
    delta_L = L_m * rel_error_L
    
    # ===== РАСЧЕТ g =====
    
    # Формула математического маятника: T = 2π√(L/g)
    # Отсюда: g = 4π²L/T²
    
    g = (4 * np.pi**2 * L_m) / (T**2)
    
    # Погрешность g
    # δg/g = √[(δL/L)² + (2δT/T)²]
    # Коэффициент 2 появляется из-за T²
    
    rel_error_g = np.sqrt(
        (delta_L / L_m)**2 + 
        (2 * delta_T / T)**2
    )
    delta_g = g * rel_error_g
    
    # ===== РЕЗУЛЬТАТЫ =====
    
    print('ИСХОДНЫЕ ДАННЫЕ:')
    print(f'  Период T = {T:.5f} +/- {delta_T:.5f} сек')
    print(f'  Относительная погрешность T: {delta_T/T*100:.3f}%')
    print(f'  Длина маятника L:')
    print(f'    В пикселях: {L_px:.2f} +/- {L_px_error:.2f} px')
    print(f'    В метрах: {L_m:.5f} +/- {delta_L:.5f} м')
    print(f'    В мм: {L_m*1000:.2f} +/- {delta_L*1000:.2f} мм')
    print(f'  Относительная погрешность L: {rel_error_L*100:.3f}%')
    
    print('' + '='*70)
    print('РЕЗУЛЬТАТ:')
    print('='*70)
    print(f'  g = {g:.4f} +/- {delta_g:.4f} м/с²')
    print(f'  g = {g:.2f} +/- {delta_g:.2f} м/с²')
    print(f'  Относительная погрешность: {rel_error_g*100:.2f}%')
    
    # Вклад различных источников
    contrib_L = (delta_L / L_m)**2
    contrib_T = (2 * delta_T / T)**2
    total_var = contrib_L + contrib_T
    
    print('ВКЛАД В ПОГРЕШНОСТЬ g:')
    print(f'  От погрешности L: {np.sqrt(contrib_L/total_var)*100:.1f}%')
    print(f'  От погрешности T: {np.sqrt(contrib_T/total_var)*100:.1f}%')
    
    # Сравнение с табличным значением
    g_table = 9.81  # м/с² (стандартное значение)
    
    print('' + '='*70)
    print('СРАВНЕНИЕ С ТАБЛИЧНЫМ ЗНАЧЕНИЕМ:')
    print('='*70)
    print(f'  Табличное g: {g_table} м/с²')
    print(f'  Измеренное g: {g:.4f} +/- {delta_g:.4f} м/с²')
    print(f'  Разница: {abs(g - g_table):.4f} м/с² ({abs(g - g_table)/g_table*100:.2f}%)')
    
    # Проверка совместимости (в пределах погрешности?)
    sigma = abs(g - g_table) / delta_g
    print(f'  Отклонение: {sigma:.2f} σ')
    
    if sigma < 1:
        print('  ✓ Результат согласуется с табличным (< 1σ)')
    elif sigma < 2:
        print('  ✓ Результат согласуется с табличным (< 2σ)')
    elif sigma < 3:
        print('  ⚠ Результат на границе согласия (< 3σ)')
    else:
        print('  ✗ Результат НЕ согласуется с табличным (> 3σ)')
    
    print('='*70)
    
    # Визуализация
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Результат с погрешностью
    ax.errorbar([1], [g], yerr=[delta_g], fmt='ro', markersize=12, 
                capsize=10, capthick=2, linewidth=2, label=f'Измерено: {g:.2f}±{delta_g:.2f}')
    
    # Табличное значение
    ax.axhline(g_table, color='blue', linestyle='--', linewidth=2, 
              label=f'Табличное: {g_table} м/с²')
    
    # Область погрешности
    ax.fill_between([0.5, 1.5], g - delta_g, g + delta_g, 
                    alpha=0.3, color='red', label='±1σ')
    
    ax.set_xlim(0.5, 1.5)
    ax.set_ylabel('g (м/с²)', fontsize=14, fontweight='bold')
    ax.set_title('Ускорение свободного падения', fontsize=16, fontweight='bold')
    ax.set_xticks([])
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print('Переменные сохранены: g, delta_g, T, L_m')